In [1]:
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from pathlib import Path


In [2]:
# 1. Project paths

BASE_DIR = Path.cwd().parent

REPORTS_DIR = BASE_DIR / "reports"

advanced_file = REPORTS_DIR / "advanced_rewards_analysis.csv"
scenario_file = REPORTS_DIR / "scenario_analysis_results.csv"

dashboard_file = REPORTS_DIR / "credit_card_dashboard.html"

In [3]:
# 2. Load analysis results

if not advanced_file.exists():
    raise FileNotFoundError(
        "Advanced rewards report not found. "
        "Run advanced_rewards_analysis.py first."
    )

if not scenario_file.exists():
    raise FileNotFoundError(
        "Scenario report not found. "
        "Run scenario_analysis.py first."
    )

advanced = pd.read_csv(advanced_file)
scenarios = pd.read_csv(scenario_file)


In [4]:
# 3. Check required columns

required_advanced = [
    "card_name",
    "annual_rewards",
    "annual_fee",
    "ongoing_net_value",
    "welcome_bonus_value",
    "first_year_net_value"
]

required_scenarios = [
    "scenario",
    "card_name",
    "ongoing_net_value"
]

for column in required_advanced:
    if column not in advanced.columns:
        raise ValueError(
            f"Missing column in advanced report: {column}"
        )

for column in required_scenarios:
    if column not in scenarios.columns:
        raise ValueError(
            f"Missing column in scenario report: {column}"
        )


In [5]:
# 4. Sort card results

advanced = advanced.sort_values(
    "first_year_net_value",
    ascending=False
)


In [6]:
# 5. Create dashboard layout

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "First-Year Net Value by Card",
        "Ongoing Annual Net Value by Card",
        "Annual Rewards, Fees and Welcome Bonuses",
        "Net Value by Spending Scenario"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

In [7]:
# 6. Chart 1: First-year net value

fig.add_trace(
    go.Bar(
        x=advanced["card_name"],
        y=advanced["first_year_net_value"],
        name="First-Year Net Value",
        text=advanced["first_year_net_value"].round(2),
        texttemplate="$%{text:,.2f}",
        textposition="outside",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "First-Year Net Value: $%{y:,.2f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)


In [8]:
# 7. Chart 2: Ongoing net value

ongoing = advanced.sort_values(
    "ongoing_net_value",
    ascending=False
)

fig.add_trace(
    go.Bar(
        x=ongoing["card_name"],
        y=ongoing["ongoing_net_value"],
        name="Ongoing Net Value",
        text=ongoing["ongoing_net_value"].round(2),
        texttemplate="$%{text:,.2f}",
        textposition="outside",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Ongoing Net Value: $%{y:,.2f}"
            "<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)


In [9]:
# 8. Chart 3: Rewards, fees and bonuses

for column, label in [
    ("annual_rewards", "Annual Rewards"),
    ("annual_fee", "Annual Fee"),
    ("welcome_bonus_value", "Welcome Bonus")
]:
    fig.add_trace(
        go.Bar(
            x=advanced["card_name"],
            y=advanced[column],
            name=label,
            hovertemplate=(
                "<b>%{x}</b><br>"
                + label + ": $%{y:,.2f}"
                + "<extra></extra>"
            )
        ),
        row=2,
        col=1
    )


In [10]:
# 9. Chart 4: Spending scenario analysis

scenario_pivot = scenarios.pivot(
    index="scenario",
    columns="card_name",
    values="ongoing_net_value"
)

for card_name in scenario_pivot.columns:
    fig.add_trace(
        go.Bar(
            x=scenario_pivot.index,
            y=scenario_pivot[card_name],
            name=card_name,
            hovertemplate=(
                "<b>%{x}</b><br>"
                + card_name
                + ": $%{y:,.2f}"
                + "<extra></extra>"
            ),
            showlegend=True
        ),
        row=2,
        col=2
    )


In [11]:
# 10. Dashboard formatting

fig.update_layout(
    title={
        "text": (
            "Credit Card Rewards Optimizer"
            "<br><sup>Rewards, fees, bonuses and spending scenarios</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    height=950,
    barmode="group",
    template="plotly_white",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": -0.20,
        "xanchor": "center",
        "x": 0.5
    },
    margin={
        "l": 60,
        "r": 40,
        "t": 110,
        "b": 140
    }
)

fig.update_yaxes(
    title_text="Annual Value ($)",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Annual Value ($)",
    row=1,
    col=2
)

fig.update_yaxes(
    title_text="Amount ($)",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Ongoing Net Value ($)",
    row=2,
    col=2
)

fig.update_xaxes(
    tickangle=-20
)


In [12]:
# 11. Export interactive HTML dashboard

fig.write_html(
    dashboard_file,
    include_plotlyjs=True,
    full_html=True
)

print("\nDashboard created successfully!")
print(f"Saved to: {dashboard_file}")

print("\nCards included:")
for card in advanced["card_name"]:
    print("-", card)

print("\nSpending scenarios included:")
for scenario in scenarios["scenario"].unique():
    print("-", scenario)


Dashboard created successfully!
Saved to: /Users/golden_horde/Desktop/FA_projects/credit-card-rewards-optimizer/reports/credit_card_dashboard.html

Cards included:
- Capital One Savor
- Chase Freedom Unlimited
- Wells Fargo Active Cash

Spending scenarios included:
- Average Consumer
- Frequent Traveler
- Gas and Commuting Consumer
- Grocery and Dining Consumer
